# Урок № 6. Файлы в операционной системе и приложение «Блокнот»

**Python Middle · 10–15 лет · Jupyter Notebook для преподавателя · 160 минут**

**Сегодняшняя миссия:** научить программу помнить текст **после закрытия окна**. Сначала разберём файлы и пути, затем напишем собственный текстовый редактор с меню.

**Связь с курсом:** в уроке № 5 мы создали окно на `tkinter`; в уроке № 6 добавим **чтение и сохранение текстовых файлов**; в уроке № 7 это пригодится для приложения «Заметки».

> **Преподавателю:** материал сохраняет все основные этапы и три файла из прикреплённой методички. После каждого нового действия есть вопрос, демонстрация или маленькая самостоятельная задача. Исправлены потенциальные проблемы с первым сохранением, отменой выбора файла, закрытием файла, русским текстом и путём к файлу.

## Как пользоваться уроком

- **Markdown** — прочитать и обсудить. **Code** — запустить сочетанием `Shift + Enter`.
- Сначала задайте детям вопрос, **что программа сделает**, потом выполните код.
- Окна Tkinter открываются **отдельно от Jupyter**. Закройте предыдущее окно, прежде чем запускать следующее.
- **Не используйте Google Colab**: он не предоставляет обычный рабочий стол для окон Tkinter. Нужен локальный Jupyter на Windows, macOS или Linux с графическим интерфейсом.
- Все учебные файлы создаются в отдельной папке `tkinter_notepad_lesson6`, чтобы не затрагивать остальные документы учеников.

**Подсказка преподавателю:** если ученик не успевает, он может только запускать готовые ячейки и изменять выделенные значения. Уверенным ученикам предложите перепечатать ключевые методы самостоятельно.

## Маршрут урока (160 минут)

| Время | Что делаем | Результат |
|---|---|---|
| 0–10 мин | Повторяем Tkinter | Вспомнили окно, поле, кнопку и обработчик |
| 10–30 мин | Windows, Linux, файлы и пути | Отличаем абсолютный путь от относительного |
| 30–55 мин | `open`, `w`, `r`, `close`, `with` | Сохранили и прочитали текст |
| 55–65 мин | Мини-практика | Закрепили запись и чтение |
| 65–85 мин | `ScrolledText`, меню | Увидели заготовку редактора |
| 85–100 мин | Перерыв / проверка предыдущих блоков | Без новых сложных понятий |
| 100–125 мин | Открытие и сохранение файлов | Настоящий работающий блокнот |
| 125–145 мин | Окно «О программе» и запуск | Проект из трёх файлов |
| 145–160 мин | Тесты, задания и итог | Каждый проверил готовое приложение |

Время — ориентир. Наиболее важная часть — **самостоятельно получить работающий блокнот**.

---
# Часть 0. Подготовка

Уроку достаточно Python 3 и встроенной библиотеки `tkinter`. Дополнительные пакеты для приложения устанавливать не нужно. Мы будем использовать стандартные модули `pathlib`, `subprocess`, `sys`.

Запустите этот служебный блок **один раз**. Он создаст папку проекта и функцию, которая открывает учебные программы в отдельных окнах.

In [ ]:
from pathlib import Path
import tkinter as tk
import subprocess
import sys

PROJECT = Path.cwd() / "tkinter_notepad_lesson6"
PROJECT.mkdir(exist_ok=True)

open_apps = []

def run_gui(filename):
    path = PROJECT / filename
    if not path.is_file():
        raise FileNotFoundError(f"Сначала создайте файл: {path}")
    process = subprocess.Popen([sys.executable, str(path)], cwd=str(PROJECT))
    open_apps.append(process)
    print(f"Открыто приложение: {filename}. Закройте его окно после проверки.")

print("Папка урока:", PROJECT)
print("Версия Python:", sys.version.split()[0])
print("Tkinter подключён успешно!")

### Повторение за 5 вопросов

1. Для чего нужен `tk.Tk()`?
2. Что делает `.title("Моё окно")`?
3. Для чего используется `.pack()`?
4. Что означает `command=...` у кнопки?
5. Что произойдёт с текстом в поле `Entry`, если закрыть программу, **не сохраняя его**?

**Важная мысль:** окно умеет показывать информацию, а **файл** позволяет сохранить её на диске и прочитать позже.

---
# Часть 1. Как устроены файлы и папки

## 1. Файл и папка — это не одно и то же

Представьте школьный рюкзак: **папка** — отдельное отделение, а **файл** — листок, лежащий внутри. Папки могут содержать другие папки — получается дерево.

- **Файл:** `домашнее_задание.txt`, `фото.png`, `проект.py`.
- **Папка:** `Школа`, `Python`, `Мои_файлы`.
- **Расширение:** часть после последней точки, которая помогает узнать тип файла.
- **Путь:** «адрес», по которому компьютер найдёт файл.

**Вопрос детям:** почему в папке могут лежать два файла `рисунок.png` и `рисунок.txt`, хотя начало имени одинаковое?

## 2. Файловая система Windows

В Windows путь часто начинается с **буквы диска** (`C:`, `D:`). В методичке приведён пример `A:\Папка2\Файл0`.

Пример знакомого адреса:

```text
C:\Users\Student\Documents\Python\test_file.txt
```

**Читаем путь:** диск `C:` → папка `Users` → папка `Student` → ... → файл `test_file.txt`.

**Важно:** в Windows для разделения частей пути используется `\`. В программном коде часто удобно использовать `pathlib.Path`, чтобы не думать о разных разделителях.

## 3. Файловая система Linux (и macOS)

В Linux пути начинаются с **корня** `/`. Папки пользователей обычно находятся внутри `/home`. На macOS пользовательские домашние каталоги обычно находятся в `/Users`.

Пример Linux:

```text
/home/student/Documents/Python/test_file.txt
```

Один и тот же файл на Windows и Linux записывается по-разному, но Python умеет работать с обоими вариантами. Поэтому сегодня мы используем `Path`.

**Блиц-опрос:** что означает `/` в самом начале Linux-пути? Что означает `C:` в начале Windows-пути?

## 4. Полный (абсолютный) и относительный пути

**Абсолютный путь** — полный адрес от корня/диска. **Относительный путь** — адрес от текущей рабочей папки программы.

Пример из методички: путь `A:\папка3\файл1` — полный, а `файл1`, когда файл лежит рядом с программой, — относительный.

Аналогия: **«ул. Лесная, дом 7»** — полный адрес; **«следующая дверь»** — относительный.

Посмотрим, где находится программа на нашем компьютере.

In [ ]:
print("Текущая рабочая папка:", Path.cwd())
print("Относительный путь:", Path("test_file.txt"))
print("Абсолютный путь:", (PROJECT / "test_file.txt").resolve())

### Мини-задание 1. Угадай путь (3 минуты)

**Устно, перед запуском следующего кода:**

1. Какой из путей абсолютный: `D:\Games\level.txt` или `level.txt`?
2. Если файл лежит в `notes/lesson.txt` рядом с программой, какой путь можно написать?
3. Почему программа может не находить относительный файл, если её запустили из другой папки?

**Ответ преподавателю:** `D:\Games\level.txt`; `notes/lesson.txt`; потому что изменилось место, откуда отсчитывается относительный путь.

In [ ]:
exercise_path = PROJECT / "my_folder"
exercise_path.mkdir(exist_ok=True)
print("Создали папку:", exercise_path)
print("Её название:", exercise_path.name)
print("Она существует?", exercise_path.exists())

---
# Часть 2. Чтение и запись файлов Python

## 5. Что делает `open()`?

`open()` открывает доступ к файлу. Для нас важны два режима **точно как в методичке**:

| Режим | Перевод | Что делает |
|---|---|---|
| `"w"` | write | Записывает файл, **заменяя** старое содержимое. Если файла нет — создаёт |
| `"r"` | read | Читает существующий файл. Если его нет — возникает ошибка |

Бонус: `"a"` (append) дописывает текст в конец, не стирая предыдущий.

Сначала выполним запись по шагам. Во всех примерах используем `encoding="utf-8"`, чтобы русские буквы сохранялись правильно.

## 6. Шаг 1 — создаём файл и записываем `hello files`

**Ожидаемый результат:** появится `test_file.txt` внутри папки урока.

- `open(..., "w")` — открывает файл для записи.
- `.write(...)` — кладёт в него текст.
- `.close()` — закрывает файл, чтобы Python завершил запись и освободил ресурс.

**Преподавателю:** сначала попросите детей предположить, создастся ли файл автоматически.

In [ ]:
filepath = PROJECT / "test_file.txt"
fileWrite = open(filepath, "w", encoding="utf-8")
fileWrite.write("hello files")
fileWrite.close()
print("Файл создан:", filepath.name)

## 7. Шаг 2 — читаем этот же файл

- `open(..., "r")` — открыть для чтения.
- `.read()` — получить текст из файла в виде строки.
- `print(...)` — напечатать прочитанное.
- `.close()` — закрыть файл после чтения.

**Вопрос:** будет ли сообщение на экране, если удалить `print(text)`?

In [ ]:
fileRead = open(filepath, "r", encoding="utf-8")
text = fileRead.read()
fileRead.close()
print(text)

### Почему в методичке перед чтением добавляют `close()`?

Если запись ещё не завершена, данные могут оставаться в буфере и не быть доступными для другого чтения так, как ожидается. Метод `close()` завершает работу с файлом и освобождает ресурсы.

**Важно:** современные ОС и Python могут вести себя по-разному, поэтому правильная привычка — **всегда закрывать файлы**. Мы не будем специально забывать `close()` в рабочем коде.

## 8. Делаем удобнее с `with`

В методичке следующий шаг — `with open(...) as ...:`. Это почти то же самое, но Python **сам закрывает** файл после выполнения вложенных строк. Обратите внимание на отступы.

**Аналогия:** взяли книгу в библиотеке на время работы, а после блока `with` она автоматически возвращается.

In [ ]:
with open(filepath, "w", encoding="utf-8") as fileWrite:
    fileWrite.write("hello files")

with open(filepath, "r", encoding="utf-8") as fileRead:
    print(fileRead.read())

### Разбираем конструкцию `with` по частям

```python
with open(filepath, "w", encoding="utf-8") as file:
    file.write("Привет!")
```

- `with` — «поработай с ресурсом и правильно заверши работу».
- `open(...)` — открывает файл.
- `as file` — имя, по которому мы обращаемся к открытомy файлу.
- Отступ перед `file.write()` показывает, что строка выполняется внутри `with`.

**Проверка понимания:** как прочитать файл, заменив `"w"` на нужную букву?

## 9. Сохраняем русские буквы и числа

Метод `.write()` принимает **строку**. Если хотим записать число `15`, сначала превращаем его в текст через `str(15)`.

Сейчас создадим новый файл, чтобы не испортить `test_file.txt` из предыдущего опыта.

In [ ]:
profile_path = PROJECT / "student.txt"
age = 15
with open(profile_path, "w", encoding="utf-8") as file:
    file.write("Привет, я изучаю Python!\n")
    file.write("Мой возраст: " + str(age) + "\n")

with open(profile_path, "r", encoding="utf-8") as file:
    print(file.read())

## 10. Небольшое расширение — режим `"a"`

**В методичке главное — `"w"` и `"r"`.** Режим `"a"` добавим как удобный бонус: он пригодится, когда нужно не стереть старый текст, а добавить ещё одну строку.

Выполните ячейку **два раза**. Что изменилось? Почему?

In [ ]:
with open(profile_path, "a", encoding="utf-8") as file:
    file.write("Я умею добавлять строки.\n")

with open(profile_path, "r", encoding="utf-8") as file:
    print(file.read())

## 11. А если файла нет?

В режиме `"r"` Python не создаёт файл. Он сообщает об ошибке `FileNotFoundError`.

Чтобы урок не прерывался красной ошибкой, используем `try/except`: **«попробуй выполнить; если файла нет, расскажи об этом»**.

In [ ]:
missing_path = PROJECT / "такого_файла_нет_12345.txt"
try:
    with open(missing_path, "r", encoding="utf-8") as file:
        print(file.read())
except FileNotFoundError:
    print("Файл не найден. Сначала его нужно создать!")

## 12. Мини-практика: «Мой дневник» (10 минут)

**Самостоятельно:** создайте файл `diary.txt`, запишите туда три строки (имя, любимый предмет, планы на завтра), затем прочитайте всё содержимое и выведите на экран.

**Условие:** для записи и чтения используйте `with`. Для самостоятельной тренировки можно поменять строки в примере ниже.

In [ ]:
# Заготовка: предложите детям изменить три текста ниже.
diary_path = PROJECT / "diary.txt"
with open(diary_path, "w", encoding="utf-8") as diary:
    diary.write("Меня зовут ...\n")
    diary.write("Мой любимый предмет — ...\n")
    diary.write("Завтра я хочу ...\n")

with open(diary_path, "r", encoding="utf-8") as diary:
    print(diary.read())

### Проверка мини-практики

- [ ] Файл `diary.txt` появился внутри папки проекта.
- [ ] В нём три строки.
- [ ] При чтении виден русский текст, а не непонятные символы.
- [ ] Рядом есть закрывающая строка блока `with` (не нужен `.close()`).

**Объяснение преподавателя:** пока мы пишем только текстовые файлы. Картинки, видео и программы тоже являются файлами, но их содержимое нельзя просто считать как обычный текст.

---
# Часть 3. Собираем «Блокнот» на Tkinter

## 13. Как будет устроено приложение?

В методичке указаны **три Python-файла**:

```text
tkinter_notepad_lesson6/
    main.py             — запускает программу
    MainWindow.py       — главное окно, меню и работа с файлами
    AboutWindow.py      — дополнительное окно «О программе»
```

В готовом Блокноте будут четыре пункта меню:

- **Открыть файл** — выбрать `.txt` и показать его текст.
- **Сохранить файл как** — выбрать имя и место для сохранения.
- **Сохранить файл** — сохранить в уже выбранный файл (или спросить имя в первый раз).
- **О программе** — показать отдельное окно с версией.

Наша цель — повторить **всю эту функциональность**.

## 14. Сначала отдельный опыт с `ScrolledText`

`Entry` из прошлого урока — поле для **одной строки**, а `ScrolledText` — большая «тетрадная страница», где можно печатать много строк и прокручивать текст.

В методичке используются размеры `width=96, height=37` и основное окно `800x600`. Сначала откроем уменьшенную демонстрацию, чтобы понять идею.

In [ ]:
%%writefile tkinter_notepad_lesson6/demo_01_window.py
import tkinter as tk
from tkinter import scrolledtext

root = tk.Tk()
root.title("Первые шаги: поле для текста")
root.geometry("600x350")

editor = scrolledtext.ScrolledText(root, width=60, height=15, font=("Arial", 12))
editor.pack(fill="both", expand=True, padx=10, pady=10)
editor.insert("1.0", "Привет! Это мой первый текстовый редактор.\n")

root.mainloop()


In [ ]:
run_gui("demo_01_window.py")

### Разбор поля `ScrolledText`

- `from tkinter import scrolledtext` — подключили готовый виджет с прокруткой.
- `scrolledtext.ScrolledText(root, ...)` — создали поле внутри окна.
- `.insert("1.0", "текст")` — добавили текст с первой строки, с первого символа.
- `.pack(fill="both", expand=True)` — поле растягивается вместе с окном.

**Вопрос:** зачем длинному документу нужна полоса прокрутки?

**Мини-задание 2:** откройте файл демонстрации (создан рядом с проектом) и поменяйте первую фразу. Затем запустите демонстрацию снова.

## 15. Учимся создавать верхнее меню

Вместо десятка отдельных кнопок удобно разместить действия в верхнем меню. По методичке используется объект `tk.Menu`, а пункты добавляются методом `.add_command()`.

**Обратите внимание:** `command=greet` без круглых скобок означает «вызови `greet`, когда нажмут пункт».

In [ ]:
%%writefile tkinter_notepad_lesson6/demo_02_menu.py
import tkinter as tk
from tkinter import messagebox

root = tk.Tk()
root.title("Как работает меню")
root.geometry("480x240")

def greet():
    messagebox.showinfo("Меню", "Вы выбрали пункт меню!")

menu = tk.Menu(root, tearoff=0)
menu.add_command(label="Нажми меня", command=greet)
menu.add_command(label="Выход", command=root.destroy)
root.config(menu=menu)

tk.Label(root, text="Открой верхнее меню", font=("Arial", 15)).pack(pady=60)
root.mainloop()


In [ ]:
run_gui("demo_02_menu.py")

### Мини-задание 3 (3 минуты)

Найдите в коде:

- Где создаётся меню?
- Где задаётся надпись пункта?
- Какой пункт вызывает `root.destroy`, то есть закрывает окно?

Попробуйте переименовать «Нажми меня» в «Привет, Python!», а затем заново запустить файл.

## 16. Готовые диалоговые окна выбора файла

Для открытия и сохранения файла не нужно рисовать собственное окно выбора папок! В `tkinter.filedialog` уже есть удобные функции:

- `askopenfilename()` — выбрать существующий файл.
- `asksaveasfilename()` — выбрать путь для сохранения.

Если человек нажал «Отмена», Python обычно получает пустую строку `""`. **Мы обязаны это проверять**, иначе программа попробует открыть файл без имени.

Следующий опыт помогает увидеть окно выбора файла. Выберите `student.txt` или `diary.txt` из папки проекта.

In [ ]:
%%writefile tkinter_notepad_lesson6/demo_03_open.py
import tkinter as tk
from tkinter import filedialog, scrolledtext
from pathlib import Path

root = tk.Tk()
root.title("Диалог выбора файла")
root.geometry("600x400")
editor = scrolledtext.ScrolledText(root, wrap="word")
editor.pack(fill="both", expand=True, padx=8, pady=8)

def choose_file():
    path = filedialog.askopenfilename(filetypes=[("Текст", "*.txt"), ("Все файлы", "*.*")])
    if not path:
        return
    text = Path(path).read_text(encoding="utf-8")
    editor.delete("1.0", tk.END)
    editor.insert("1.0", text)

tk.Button(root, text="Выбрать файл", command=choose_file).pack(pady=8)
root.mainloop()


In [ ]:
run_gui("demo_03_open.py")

### Проверка понимания

**Вопрос 1:** что возвращает `askopenfilename()` — текст файла или его адрес?

**Ответ:** адрес выбранного файла. Текст нужно прочитать отдельно.

**Вопрос 2:** зачем в обработчике перед вставкой нового текста нужен `editor.delete("1.0", tk.END)`?

**Ответ:** иначе новый документ добавится к старому вместо замены.

**Примечание:** в этой маленькой демонстрации мы не обрабатываем ошибки формата файла. В основном проекте добавим `try/except`.

---
# Часть 4. Пишем проект из трёх файлов

## 17. Файл № 1 — `AboutWindow.py`

Начнём с маленького окна «О программе». Этот файл можно создать первым — он не зависит от остальных файлов проекта.

**Что такое класс?** Представьте чертёж комнаты. Класс `About` — чертёж окошка, а `About(self)` позже создаст конкретный экземпляр.

- `tk.Toplevel` — отдельное окно внутри приложения.
- `super().__init__(parent)` — запустить стандартные настройки родительского класса.
- `tk.Label` — надпись.
- `tk.Button(..., command=self.destroy)` — кнопка «Закрыть».

In [ ]:
%%writefile tkinter_notepad_lesson6/AboutWindow.py
import tkinter as tk


class About(tk.Toplevel):
    """Небольшое окно с информацией о нашей программе."""

    def __init__(self, parent):
        super().__init__(parent)
        self.title("О программе")
        self.geometry("300x160")
        self.resizable(False, False)

        label = tk.Label(self, text="Блокнот на Python\nВерсия: 1.0", font=("Arial", 13))
        label.pack(padx=20, pady=22)

        button = tk.Button(self, text="Закрыть", command=self.destroy)
        button.pack(pady=5, ipadx=10, ipady=3)


### Проверим файл «О программе» отдельно

Пока главное окно ещё не готово. Поэтому откроем родительское пустое окно и создадим рядом `About`. Это **демонстрация**, не дополнительный файл проекта.

In [ ]:
# Создаём временную демонстрацию в отдельном файле.
about_demo = '''import tkinter as tk
from AboutWindow import About

root = tk.Tk()
root.title("Проверка окна справки")
root.geometry("360x200")
tk.Button(root, text="О программе", command=lambda: About(root)).pack(pady=55)
root.mainloop()
'''
(PROJECT / "demo_04_about.py").write_text(about_demo, encoding="utf-8")
run_gui("demo_04_about.py")

## 18. Файл № 2 — `MainWindow.py`: основное окно

Теперь создадим главное окно и поле для текста. В методичке `MainWindow` наследуется от `tk.Tk`.

**Строки, которые важно запомнить:**

- `class MainWindow(tk.Tk):` — делаем свой вид окна.
- `def __init__(self):` — что происходит при его создании.
- `super().__init__()` — сначала создаём обычное окно Tkinter.
- `self._filepath = None` — путь пока неизвестен (это исправляет проблему самого первого сохранения).
- `self._create_menu()` — создадим меню отдельным методом, **который добавим следующим блоком**.

⚠️ **Не запускайте `MainWindow.py` отдельно сейчас:** пока мы дописываем класс, методы для меню ещё не готовы. По окончании всех этапов он заработает.

In [ ]:
%%writefile tkinter_notepad_lesson6/MainWindow.py
import tkinter as tk
from tkinter import scrolledtext, filedialog, messagebox
from AboutWindow import About


class MainWindow(tk.Tk):
    """Главное окно текстового редактора."""

    def __init__(self):
        super().__init__()
        self.title("Блокнот")
        self.geometry("800x600")
        self._filepath = None  # Пока пользователь не выбрал файл

        self.edit_text = scrolledtext.ScrolledText(
            self, width=96, height=37, wrap="word", font=("Arial", 12)
        )
        self._create_menu()
        self.edit_text.pack(fill="both", expand=True, padx=8, pady=8)


## 19. Добавляем метод `_create_menu`

Этот блок **дописывает** метод внутрь класса `MainWindow` в файл `MainWindow.py`. Флаг `-a` в команде `%%writefile -a` означает «добавить к уже созданному файлу».

В методичке ровно четыре нужных действия: открытие, сохранение как, обычное сохранение, справка.

**Спросите перед запуском:** почему при `command=self._open_file` нет `()`?

**Ответ:** мы передаём сам обработчик — меню вызовет его при нажатии.

In [ ]:
%%writefile -a tkinter_notepad_lesson6/MainWindow.py

    def _create_menu(self):
        self._menu = tk.Menu(self, tearoff=0)
        self._menu.add_command(label="Открыть файл", command=self._open_file)
        self._menu.add_command(label="Сохранить файл как", command=self._save_file_as)
        self._menu.add_command(label="Сохранить файл", command=self._save_file)
        self._menu.add_command(label="О программе", command=self._open_about)
        self.config(menu=self._menu)


## 20. Добавляем метод `_open_file` — открываем текстовый файл

Проследите путь данных:

**Человек выбирает файл → `filedialog` возвращает его путь → `open(..., "r")` читает текст → `delete` очищает поле → `insert` показывает новое содержимое.**

Как и в методичке, сохраняем путь в `_filepath` и меняем заголовок окна.

Для устойчивости добавим:

- `if not path: return` — безопасное действие при «Отмена».
- `encoding="utf-8"` — русский текст.
- `try/except` — сообщение вместо падения, если файл не прочитался.

**Мини-задание 4:** найдите строку, которая удаляет старый текст из редактора. Что будет, если её убрать?

In [ ]:
%%writefile -a tkinter_notepad_lesson6/MainWindow.py

    def _open_file(self):
        path = filedialog.askopenfilename(
            title="Выберите текстовый файл",
            filetypes=[("Текстовые файлы", "*.txt"), ("Все файлы", "*.*")],
        )
        if not path:  # Нажали «Отмена»
            return

        try:
            with open(path, "r", encoding="utf-8") as file:
                text = file.read()
        except (OSError, UnicodeError) as error:
            messagebox.showerror("Ошибка открытия", str(error))
            return

        self.edit_text.delete("1.0", tk.END)
        self.edit_text.insert("1.0", text)
        self._filepath = path
        self.title(f"Блокнот — {path}")


### Сложная запись `"1.0"` — что она означает?

У виджета `Text`/`ScrolledText` свои координаты текста:

- `"1.0"` — первая строка, нулевой (первый по счёту) символ.
- `tk.END` — конец всего текста.
- `.delete("1.0", tk.END)` — очистить полностью.
- `.insert("1.0", text)` — вставить текст с начала.
- `.get("1.0", "end-1c")` — получить текст **без служебного дополнительного перевода строки** в конце.

Проверьте на уже знакомой строке Python: индекс 0 — это **первый** символ, а не последний.

## 21. Добавляем метод `_save_file_as`

Это кнопка «**Сохранить файл как**». Она спрашивает, **в какую папку** и **под каким именем** записать содержимое.

Мы заранее запоминаем старый адрес файла: если запись не получится (например, нет прав), можно вернуть его обратно.

**Важно для детей:** выбрать место сохранения — ещё не значит записать содержимое; для этого у нас будет метод `_write_file()`.

In [ ]:
%%writefile -a tkinter_notepad_lesson6/MainWindow.py

    def _save_file_as(self):
        path = filedialog.asksaveasfilename(
            title="Сохранить файл как",
            defaultextension=".txt",
            filetypes=[("Текстовые файлы", "*.txt"), ("Все файлы", "*.*")],
        )
        if not path:  # Нажали «Отмена»
            return
        previous_path = self._filepath
        self._filepath = path
        if not self._write_file():
            self._filepath = previous_path


## 22. Добавляем `_save_file` и `_write_file`

Здесь мы выполняем требование методички — обычное сохранение и сохранение «как» различаются:

- **Сохранить:** если путь уже известен — записать туда сразу.
- **Сохранить (первый раз):** если адрес ещё не выбран — открыть «Сохранить как».
- **Сохранить как:** всегда спросить имя и место нового файла.

Функцию записи мы выделяем в отдельный метод `_write_file`, чтобы не повторять одинаковые строки. Это небольшое педагогическое улучшение по сравнению с исходным примером.

**Задайте детям вопрос:** почему при записи нужно `"w"`, а при чтении `"r"`?

In [ ]:
%%writefile -a tkinter_notepad_lesson6/MainWindow.py

    def _save_file(self):
        if self._filepath is None:
            self._save_file_as()  # Первое сохранение: просим имя файла
            return
        self._write_file()

    def _write_file(self):
        text = self.edit_text.get("1.0", "end-1c")
        try:
            with open(self._filepath, "w", encoding="utf-8") as file:
                file.write(text)
        except OSError as error:
            messagebox.showerror("Ошибка сохранения", str(error))
            return False
        self.title(f"Блокнот — {self._filepath}")
        return True


## 23. Добавляем `_open_about` — окно справки

В методичке последний обработчик создаёт объект `About(self)` и делает `about.grab_set()`.

- `About(self)` — открывает дополнительное окно «О программе».
- `.grab_set()` — переводит ввод в это окно, пока его не закроют.

Это связывает `MainWindow.py` и `AboutWindow.py`.

In [ ]:
%%writefile -a tkinter_notepad_lesson6/MainWindow.py

    def _open_about(self):
        about = About(self)
        about.grab_set()  # Пока окно открыто, пользователь работает с ним


### Проверяем, какие файлы уже появились

Все методы основного окна теперь дописаны. Сначала посмотрим на содержимое каталога, потом создадим `main.py`.

In [ ]:
for filename in sorted(PROJECT.glob("*.py")):
    print(filename.name)

## 24. Файл № 3 — `main.py`: вход в программу

Этот файл очень короткий, но важный:

- `from MainWindow import MainWindow` — подключаем главный класс.
- `if __name__ == "__main__":` — выполнять запуск, когда файл запускают как программу.
- `app = MainWindow()` — создать наше главное окно.
- `app.mainloop()` — ждать действий человека: печати, кликов и выбора пунктов меню.

**Это последняя обязательная часть исходной методички.**

In [ ]:
%%writefile tkinter_notepad_lesson6/main.py
from MainWindow import MainWindow


if __name__ == "__main__":
    app = MainWindow()
    app.mainloop()


---
# Часть 5. Запускаем и тестируем готовый «Блокнот»

## 25. Сначала проверяем синтаксис проекта

Этот служебный блок **не открывает окон**. Он проверяет, что три Python-файла имеют правильный синтаксис, а также что наши файлы действительно находятся в каталоге урока.

При успехе должны появиться три сообщения `OK`.

In [ ]:
import ast

for filename in ["AboutWindow.py", "MainWindow.py", "main.py"]:
    path = PROJECT / filename
    ast.parse(path.read_text(encoding="utf-8"), filename=filename)
    print("OK:", filename)

## 26. Настоящий запуск

Запустите следующий блок. **В отдельном окне** появится наш «Блокнот».

**Важное правило:** на учебных компьютерах убедитесь, что пользователь может создавать файлы в выбранной им папке. Для тестирования удобно сохранять документы в папку урока или «Документы».

In [ ]:
run_gui("main.py")

## 27. Чек-лист проверки — выполняем прямо в приложении (10 минут)

- [ ] Открылось окно с названием «Блокнот» размером около 800×600.
- [ ] В центральное поле можно напечатать несколько строк.
- [ ] Поле прокручивается, когда текста много.
- [ ] Меню «**Сохранить файл**» при первом использовании предлагает выбрать имя.
- [ ] Создан файл `мой_текст.txt` в папке, которую выбрал ученик.
- [ ] После изменения текста второй клик «**Сохранить файл**» обновляет **тот же** файл.
- [ ] «**Сохранить файл как**» позволяет создать копию под новым именем.
- [ ] «**Открыть файл**» загружает ранее сохранённый текст.
- [ ] «**О программе**» открывает отдельное окно «Версия: 1.0».
- [ ] Кнопка «**Закрыть**» убирает окно справки.
- [ ] «Отмена» в окне выбора файла не вызывает ошибку.

**Преподавателю:** пусть дети сначала сохранят **русскую** фразу, затем закроют приложение, запустят его заново и откроют тот же файл. Это лучший способ показать, зачем понадобилась работа с файлами.

## 28. Делаем эксперимент: что сохраняется, а что нет?

1. Напечатайте в программе: «Я научился писать программы с файлами!».
2. Нажмите «Сохранить» и выберите имя `мой_первый_блокнот.txt`.
3. Закройте окно.
4. Запустите снова `main.py`.
5. Откройте ранее сохранённый файл.

**Вывод:** само поле на экране после закрытия забывает текст; но содержимое **файла** остаётся и снова загружается благодаря `open(..., "r")`.

---
# Часть 6. Практика на закрепление

## 29. Упражнения: обязательный уровень

**Задание A. Мой стиль (3 минуты).** Измените название «Блокнот» на «Блокнот [ваше имя]» в `MainWindow.py` и запустите заново.

**Задание Б. Размер и шрифт (4 минуты).** Попробуйте заменить `800x600` на `700x500`, а размер шрифта с `12` на `14`.

**Задание В. Собственный документ (5 минут).** Запишите три строки о сегодняшнем уроке, сохраните файл и откройте повторно после перезапуска.

**Задание Г. Проверка отмены (2 минуты).** Выберите «Открыть файл» → «Отмена». Объясните, почему редактор не закрылся.

**Важно:** во время повторного выполнения ячеек `%%writefile` проект перезаписывается. Для самостоятельных изменений лучше редактировать созданные `.py`-файлы в редакторе кода **после** прохождения всех ячеек урока.

## 30. Дополнительные задания для тех, кто закончил раньше

- **Лёгкое:** добавьте пункт меню «Очистить текст», который вызывает `self.edit_text.delete("1.0", tk.END)`.
- **Среднее:** добавьте отдельный пункт меню «Новый файл»: очищает поле, ставит `self._filepath = None`, возвращает заголовок «Блокнот». Предупреждайте пользователя заранее, что несохранённый текст будет потерян.
- **Среднее:** добавьте `messagebox.showinfo("Успех", "Файл сохранён!")` после успешной записи.
- **Сложнее (необязательно):** исследуйте `messagebox.askyesno` и подумайте, как защитить несохранённый текст от случайного закрытия.

Эти задания не требуются исходной методичкой — они для расширенной практики.

## 31. Типичные ошибки и способы их исправить

| Симптом | Что проверить |
|---|---|
| `ModuleNotFoundError: No module named 'AboutWindow'` | Файл должен называться `AboutWindow.py` и находиться рядом с `MainWindow.py` |
| `ModuleNotFoundError: No module named 'MainWindow'` | Создали ли `MainWindow.py`? Не меняли ли регистр букв? |
| `AttributeError: '_open_file'` | Все ячейки с `%%writefile -a` должны быть выполнены **по одному разу и по порядку** |
| Пункт «Сохранить» ничего не делает | Проверьте `_filepath`, вызов `_save_file` и выбранный путь |
| Файл не появился | Убедитесь, что нажали «Сохранить», а не «Отмена» |
| В тексте странные символы | Для русских букв используйте `encoding="utf-8"` |
| После исправления кода ничего не меняется | Закройте старое окно и запустите `main.py` заново |
| Нет окна, хотя ошибок нет | Tkinter требуется локальный графический рабочий стол |
| Не удаётся открыть файл | Проверьте, существует ли файл и является ли он текстовым |

**Преподавателю:** если методы случайно добавились в `MainWindow.py` дважды, выполните ячейку базового файла `MainWindow.py`, затем по порядку все пять ячеек с `%%writefile -a`.

## 32. Вопросы для финального обсуждения

1. Чем отличается файл от папки?
2. В чём разница файловой системы Windows и Linux?
3. Что такое абсолютный путь и что такое относительный?
4. Чем отличается режим `"w"` от режима `"r"`?
5. Для чего нужен `.close()`, и почему чаще удобно писать `with`?
6. Зачем нашему Блокноту `ScrolledText`, а не `Entry`?
7. Что делает `filedialog.askopenfilename()`?
8. Где программа запоминает адрес последнего выбранного файла?
9. Что делает метод `grab_set()`?
10. Чем отличаются `main.py`, `MainWindow.py` и `AboutWindow.py`?

**Ожидаемый результат:** ребёнок может своими словами объяснить путь данных: **файл → чтение → поле текста → редактирование → запись в файл**.

## 33. Короткое домашнее задание

**Обязательно:** запустите готовый Блокнот, напечатайте небольшой текст (5–6 строк), сохраните его как `homework.txt`, закройте программу, откройте её снова и загрузите файл.

**По желанию:** измените цвет фона поля редактора или добавьте пункт «Очистить текст».

**Что показать преподавателю на следующем уроке:** рабочую программу, сохранённый `.txt` и умение объяснить `with open(..., "w")` и `with open(..., "r")`.

## 34. Служебный блок — закрыть открытые учебные приложения

В конце занятия при необходимости завершите все окна, которые были запущены из ячеек `run_gui()`. Сначала предложите детям закрыть программы обычной кнопкой `×` — это полезная привычка. Данный блок нужен только если остались забытые окна.

In [ ]:
for process in open_apps:
    if process.poll() is None:
        process.terminate()
print("Запущенные из ноутбука окна завершены (если были ещё открыты).")

---
### Методическая сверка с исходным уроком № 6

**Сохранено из методички:** дерево папок Windows и Linux, полные/относительные пути, запись через `open(..., "w")`, чтение через `open(..., "r")`, закрытие `.close()` и переход к `with`, а также трёхфайловый проект `MainWindow.py` + `AboutWindow.py` + `main.py` с классами `MainWindow`/`About`, `ScrolledText`, меню из четырёх команд, открытием/сохранением файла и отдельным окном справки.

**Педагогические дополнения:** мини-задания, пояснения сложных выражений, примеры с Unicode, обработка «Отмена», ошибки чтения/записи, проверка работоспособности и дополнительные упражнения. Это учебный текстовый редактор: он предназначен для простых **UTF-8 текстовых файлов**, а не для любых форматов вроде Word/PDF.